In [ ]:
!pip install -q langgraph langchain-groq

In [ ]:
from langchain_groq import ChatGroq

# Replace with your own Groq API Key
GROQ_API_KEY = " "

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
from typing import TypedDict

class TeamState(TypedDict):
    task: str
    worker_result: str
    summary: str

In [ ]:
def worker(state: TeamState):
    response = llm.invoke(
        "Solve this math problem. Return only the final number.\n\n"
        + state["task"]
    )

    return {
        "worker_result": response.content.strip()
    }


def supervisor(state: TeamState):
    response = llm.invoke(
        f"""
The worker solved this problem:

{state['task']}

Worker's answer:
{state['worker_result']}

Write a short one-line summary.
"""
    )

    return {
        "summary": response.content.strip()
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TeamState)

builder.add_node("worker", worker)
builder.add_node("supervisor", supervisor)

builder.add_edge(START, "worker")
builder.add_edge("worker", "supervisor")
builder.add_edge("supervisor", END)

graph = builder.compile()

In [ ]:
result = graph.invoke(
    {
        "task": "What is 144 divided by 12, then plus 5?"
    }
)

print("Worker Result:")
print(result["worker_result"])

print("\nSupervisor Summary:")
print(result["summary"])

Worker Result:
12 * 12 = 144
144 / 12 = 12
12 + 5 = 17

Supervisor Summary:
The worker solved the problem by first finding the square of 12 (144), then dividing it by 12 to get 12, and finally adding 5 to get the answer of 17.
